# COVID-19 cases vs population (country-year)
Uses only the Python standard library so it runs offline with no packages.

1. Sum daily `new_cases` to calendar year.
2. Drop aggregates (`OWID_WRL`, `WLD`, `SAS`, blank ISO, region names).
3. Inner-join World Bank `SP.POP.TOTL` on ISO-3 + year (one-to-one).

Replace sample paths with the OWID compact CSV and the World Bank population ZIP after download. Population is a mid-year estimate. Result grain is **country-year**.


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

root = Path(".")
DROP = {"OWID_WRL", "WLD", "SAS", "EUU", "SSF", "EAS", "LCN", "MEA", "NAC"}
NAME_DROP = {"world", "africa", "asia", "europe", "south asia"}

def keep(iso, name=""):
    code = (iso or "").strip().upper()
    if len(code) != 3 or code in DROP or code.startswith("OWID_"):
        return False
    return str(name).strip().lower() not in NAME_DROP

cases = defaultdict(float)
with (root / "data/samples/covid-cases-sample.csv").open() as fh:
    for row in csv.DictReader(fh):
        if not keep(row["iso_code"], row["location"]):
            continue
        year = int(row["date"][:4])
        cases[(row["iso_code"].upper(), year)] += float(row["new_cases"] or 0)

pop = {}
with (root / "data/samples/wb-population-sample.csv").open() as fh:
    for row in csv.DictReader(fh):
        if not keep(row["countryiso3code"], row["country"]):
            continue
        pop[(row["countryiso3code"].upper(), int(row["date"]))] = float(row["value"])

assert ("OWID_WRL", 2020) not in cases
assert ("WLD", 2020) not in pop
joined = []
for key, new_cases in sorted(cases.items()):
    if key not in pop:
        continue
    joined.append((key[0], key[1], new_cases, pop[key], new_cases / pop[key] * 1e5))
keys = [r[:2] for r in joined]
assert len(keys) == len(set(keys)), "join is not one-to-one"
assert joined, "no overlapping country-years"
print("iso year cases population cases_per_100k")
for row in joined:
    print(*row)
print(f"{len(joined)} country-year rows after dropping aggregates")
